# Lab: The camera's point of view

You will write the three turns about x, y and z, build the camera's mount as one 4×4 matrix, and send a pixel's ray
down to the road. Then you swap your mount into the simulator's camera and drive with it.
Read the lecture first; this lab points back to its sections by title and follows its order.

Run the cells from top to bottom (Shift+Enter). Each problem has a "Guess first" line: write your guess down before you
run the cell, then compare. `practice` prints a green tick or tells you what it expected.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/result_first.png" alt="Top: one camera picture from the first bend of gentle, drawn by the simulator. Bottom: the same moment drawn through the world_to_ego you wrote and the mount you write in this lesson. In 18 pictures sampled along the lap, not one of the 57,600 pixels differs, and the lap matches step for step: 1778 steps, and a mean distance from the lane centre of 0.376 m. That 0.376 m is the provided car&#x27;s own error: with the simulator&#x27;s own camera it takes the same 1778 steps and gets the same 0.376 m, so a correct camera adds none.">
<figcaption><em>Lecture figure 1. Top: one camera picture from the first bend of gentle, drawn by the simulator. Bottom: the same moment drawn through the world_to_ego you wrote and the mount you write in this lesson. In 18 pictures sampled along the lap, not one of the 57,600 pixels differs, and the lap matches step for step: 1778 steps, and a mean distance from the lane centre of 0.376 m. That 0.376 m is the provided car&#x27;s own error: with the simulator&#x27;s own camera it takes the same 1778 steps and gets the same 0.376 m, so a correct camera adds none.</em></figcaption>
</figure>

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q "git+https://github.com/vvknyn/self-driving-ai-course@main"
%matplotlib inline

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from zero2fsd.car import Car, run
from zero2fsd.dashboard import dashboard
from zero2fsd.grade import check, practice
from zero2fsd.sim import BicycleState, Camera, make_scenario
from zero2fsd.sim.scenarios import TARGET_SPEED

The next cell brings in the helpers this lab builds on. `invert_pose` is yours from *Poses you can chain*: to use your own,
replace the import with your function (it must keep the name). `pixel_ray` is provided. Exercise B also needs your
`world_to_ego` from *The view from the driver's seat*; it is brought in next to that exercise.

In [ ]:
from zero2fsd.geom import invert_pose, pixel_ray

camera = Camera()
print(f"f = {camera.f:.0f} px, centre ({camera.cx}, {camera.cy}), height {camera.mount_height} m, pitch 5 degrees = {math.radians(5):.4f} rad")

## 1. A third axis, and the turn about each

The ego frame gets a third axis, z, up (lecture: "A third axis"). A positive turn about an axis follows the right-hand
rule: thumb along the axis, fingers curl the way the turn goes (lecture: "Turning about each axis").

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/ego_frame_3d.png" alt="The ego frame drawn in 3D on the car: x forward along the car, y to its left and z up, all three from the origin on the road under the front axle. The lens sits on the z axis at (0, 0, 1.4). A curved arrow shows how the right hand&#x27;s fingers curl from x toward y; the thumb would point along z, up.">
<figcaption><em>Lecture figure 2. The ego frame drawn in 3D on the car: x forward along the car, y to its left and z up, all three from the origin on the road under the front axle. The lens sits on the z axis at (0, 0, 1.4). A curved arrow shows how the right hand&#x27;s fingers curl from x toward y; the thumb would point along z, up.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/right_hand_turn.png" alt="Looking down the z axis from above: the right thumb would point up out of the page, and a curved arrow shows how the fingers curl from x, forward, toward y, left. The curved arrow is the positive turn about z, a left turn, the direction of a positive yaw θ.">
<figcaption><em>Lecture figure 3. Looking down the z axis from above: the right thumb would point up out of the page, and a curved arrow shows how the fingers curl from x, forward, toward y, left. The curved arrow is the positive turn about z, a left turn, the direction of a positive yaw θ.</em></figcaption>
</figure>

**Problem 1.** Write the three turns. `rot_z` is the 2D rotation you know, with a 1 in the corner; `rot_x` and `rot_y`
put the same cosines and sines in the other two planes. Mind where the minus sign goes in `rot_y`: its pattern is
not the same as the other two.

**Guess first:** `rot_y(0.0873) @ [1, 0, 0]` (a 5° turn about the car's left-pointing y axis, applied to forward):
does the nose point a little up or a little down?

In [ ]:
def rot_x(a):
    return np.eye(3)  # your code: [[1, 0, 0], [0, cos, -sin], [0, sin, cos]]


def rot_y(a):
    return np.eye(3)  # your code


def rot_z(a):
    return np.eye(3)  # your code


from zero2fsd.geom import rotation_matrix

x, y, z = np.eye(3)
pitch = math.radians(5.0)
print("forward x left =", np.cross(x, y), "(up)")
practice("rot_z(pi/2) turns forward to left", rot_z(math.pi / 2) @ x, (0, 1, 0), tol=1e-12)
practice("rot_x(pi/2) turns left to up", rot_x(math.pi / 2) @ y, (0, 0, 1), tol=1e-12)
practice("rot_y(5 deg) tips forward down", rot_y(pitch) @ x, (math.cos(pitch), 0.0, -math.sin(pitch)), tol=1e-12)
practice("rot_z's top-left 2 by 2 is the 2D turn", rot_z(0.591)[:2, :2], rotation_matrix(0.591), tol=1e-12)

A positive turn about y tips the nose **down**: forward becomes (0.9962, 0, −0.0872). If you put the minus sign on the
other `sin` you get (0.9962, 0, +0.0872): a camera tipped up. The first column of `rot_y` is where forward goes; read
the columns of any rotation that way.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/three_rotations.png" alt="Three panels stacked, one per axis, each with a curved arrow for the positive turn: the fingers of the right hand curl that way when the thumb points along the axis. The pitch panel is seen from the car&#x27;s left side, so forward points left; the other side views in this lesson are from the right. About x, roll: left tips toward up. About y, pitch: forward tips toward down. About z, yaw: forward turns toward left.">
<figcaption><em>Lecture figure 4. Three panels stacked, one per axis, each with a curved arrow for the positive turn: the fingers of the right hand curl that way when the thumb points along the axis. The pitch panel is seen from the car&#x27;s left side, so forward points left; the other side views in this lesson are from the right. About x, roll: left tips toward up. About y, pitch: forward tips toward down. About z, yaw: forward turns toward left.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/pitch_columns.png" alt="The columns of the pitch matrix drawn as arrows, seen from the car&#x27;s right side. Forward, (1, 0, 0), turns to (0.9962, 0, −0.0872), dipping below level; up, (0, 0, 1), turns to (0.0872, 0, 0.9962), leaning forward; left, (0, 1, 0), points into the page and does not move. The 5 degree turn is drawn 3 times larger than it is.">
<figcaption><em>Lecture figure 5. The columns of the pitch matrix drawn as arrows, seen from the car&#x27;s right side. Forward, (1, 0, 0), turns to (0.9962, 0, −0.0872), dipping below level; up, (0, 0, 1), turns to (0.0872, 0, 0.9962), leaning forward; left, (0, 1, 0), points into the page and does not move. The 5 degree turn is drawn 3 times larger than it is.</em></figcaption>
</figure>

## 2. The camera's own axes

The camera's frame is not the ego frame: x right across the picture, y down it, z out through the lens (lecture:
"The camera's own axes"). `B` takes ego numbers to camera numbers; `Bᵀ` takes them back.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/optical_vs_ego.png" alt="Two panels, stacked, at the lens of a level camera. Top: the ego frame, x forward, y left, z up. Bottom: the optical frame, x right, y down, z forward, out through the lens. The camera&#x27;s x is the car&#x27;s −y, its y is the car&#x27;s −z, and its z is the car&#x27;s x.">
<figcaption><em>Lecture figure 6. Two panels, stacked, at the lens of a level camera. Top: the ego frame, x forward, y left, z up. Bottom: the optical frame, x right, y down, z forward, out through the lens. The camera&#x27;s x is the car&#x27;s −y, its y is the car&#x27;s −z, and its z is the car&#x27;s x.</em></figcaption>
</figure>

**Guess first:** a level camera looks along ego forward. What does `Bᵀ` do to the camera's z axis (0, 0, 1)?

In [ ]:
B = np.array([[0.0, -1.0, 0.0], [0.0, 0.0, -1.0], [1.0, 0.0, 0.0]])  # ego (forward, left, up) -> camera (right, down, forward)
print("B @ forward, left, up =", B @ x, B @ y, B @ z)
print("B.T @ camera z axis   =", B.T @ [0, 0, 1], "(ego forward: a level camera looks straight ahead)")

## 3. One 4×4 for the mount

The camera sits 1.4 m up and tipped down by φ = 5°. Its turn is `rot_y(φ) @ Bᵀ`, in that order: first `Bᵀ` lines
the camera's axes up with the car's, then the pitch tips the whole camera (lecture: "One 4×4 for the mount"). The other order
gives a camera looking level and 5° to the right.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/wrong_order.png" alt="Two cameras&#x27; lens axes in two stacked panels: top, seen from above; bottom, seen from the car&#x27;s right side. The correct camera, R_y(φ) times B transpose, looks straight ahead in the top panel and 5 degrees down in the bottom one. The wrong one, B transpose times R_y(φ), looks 0.0873 rad to the car&#x27;s right in the top panel and level in the bottom one. The angles are drawn 3 times larger than they are.">
<figcaption><em>Lecture figure 7. Two cameras&#x27; lens axes in two stacked panels: top, seen from above; bottom, seen from the car&#x27;s right side. The correct camera, R_y(φ) times B transpose, looks straight ahead in the top panel and 5 degrees down in the bottom one. The wrong one, B transpose times R_y(φ), looks 0.0873 rad to the car&#x27;s right in the top panel and level in the bottom one. The angles are drawn 3 times larger than they are.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/mount_side.png" alt="The mount seen from the car&#x27;s right side, not to scale: the lens 1.4 m above the ego origin, its axis dipping 5 degrees below level and meeting the road 16.0 m ahead. The optical frame&#x27;s z and y arrows are drawn at the lens, and the ego frame&#x27;s x and z arrows at the origin.">
<figcaption><em>Lecture figure 8. The mount seen from the car&#x27;s right side, not to scale: the lens 1.4 m above the ego origin, its axis dipping 5 degrees below level and meeting the road 16.0 m ahead. The optical frame&#x27;s z and y arrows are drawn at the lens, and the ego frame&#x27;s x and z arrows at the origin.</em></figcaption>
</figure>

**Problem 2.** Build the 4×4 `T_ego_cam(height_m, pitch_rad)` in the next section, then project 200 seeded road
points through `invert_pose(T_ego_cam(...))` and the pinhole step, and compare with the simulator's own camera.

## 4. Graded exercises

### Exercise A: `T_ego_cam` (Exercise B uses it)

`T_ego_cam(height_m, pitch_rad)` returns the camera's pose in the ego frame as a 4×4 array: the turn
`rot_y(pitch_rad) @ Bᵀ` in the top-left 3×3 block, the lens position (0, 0, `height_m`) in the last column, and
`0 0 0 1` below. A positive pitch tips the nose down.

Exercise A checks that your 4×4 is a pure turn plus a shift (last row `0 0 0 1`), that it equals the reference entry by
entry at three settings, that the lens axis points where it should, and that 200 seeded road points land within
$10^{-6}$ px of where the simulator draws them. Exercise B takes the pair `(world_to_ego, T_ego_cam)`: it checks your mount as Exercise A does, then draws
18 sampled camera pictures through your two functions (lecture: "A camera mounted wrong"); none may differ from the
simulator's, and the provided car must complete `gentle`.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/projected_points.png" alt="A camera picture of the empty road with 195 of the 200 random road points, the ones that fall inside the 320 by 180 picture, drawn twice: open circles where the renderer&#x27;s ground_to_pixel puts them, and dots where your 4×4 and the pinhole step put them. Every dot sits in the centre of its circle.">
<figcaption><em>Lecture figure 9. A camera picture of the empty road with 195 of the 200 random road points, the ones that fall inside the 320 by 180 picture, drawn twice: open circles where the renderer&#x27;s ground_to_pixel puts them, and dots where your 4×4 and the pinhole step put them. Every dot sits in the centre of its circle.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/blind_zone.png" alt="The mount from the car&#x27;s right side, not to scale. The ray through the bottom row of the picture leaves the lens and meets the road 2.06 m ahead, and the lens axis meets it 16.0 m ahead. The road from the ego origin to 2.06 m ahead is shaded grey: no pixel sees it. To be seen, the ego origin would need row 1918, and the picture ends at row 179.">
<figcaption><em>Lecture figure 10. The mount from the car&#x27;s right side, not to scale. The ray through the bottom row of the picture leaves the lens and meets the road 2.06 m ahead, and the lens axis meets it 16.0 m ahead. The road from the ego origin to 2.06 m ahead is shaded grey: no pixel sees it. To be seen, the ego origin would need row 1918, and the picture ends at row 179.</em></figcaption>
</figure>

In [ ]:
from zero2fsd.geom import CAM_FROM_EGO_AXES as B, rot_y  # swap in your own rot_y from Problem 1 and the B above


def T_ego_cam(height_m, pitch_rad):
    """The camera's pose in the ego frame, as a 4 by 4 array.

    Input:   height_m, how high the lens is above the road (m); pitch_rad, how far the camera tips DOWN (rad).
    Output:  T, shape (4, 4): T[:3, :3] = rot_y(pitch_rad) @ B.T, T[:3, 3] = (0, 0, height_m), last row 0 0 0 1.
    Example: T_ego_cam(1.4, math.radians(5))
             -> [[0, -0.0872, 0.9962, 0], [-1, 0, 0, 0], [0, -0.9962, -0.0872, 1.4], [0, 0, 0, 1]]
    """
    raise NotImplementedError

**Guess first (Exercise A):** will your pixels match the simulator's to $10^{-6}$ px, or only to about 0.1 px?

In [ ]:
check("1.1.5.a", T_ego_cam)

A correct mount matches to 0.0 px: it is the same arithmetic the simulator uses. A camera tipped **up** by mistake
(`rot_y(-pitch_rad)`) puts the same points about 40 px away, and the check tells you the lens axis points up.

The same projection by hand, for 200 seeded road points:

In [ ]:
rng = np.random.default_rng(0)
forward, left = rng.uniform(2.0, 60.0, 200), rng.uniform(-8.0, 8.0, 200)
ground = np.stack([forward, left, np.zeros(200)], axis=1)
try:
    mount = T_ego_cam(1.4, math.radians(5))
    if not np.array_equal(mount[3], (0.0, 0.0, 0.0, 1.0)):
        print(f"the last row of your 4×4 is {mount[3]}; it must be 0 0 0 1, or the shift is lost")
    p_cam = (invert_pose(mount) @ np.c_[ground, np.ones(200)].T).T[:, :3]
    u = camera.cx + camera.f * p_cam[:, 0] / p_cam[:, 2]
    v = camera.cy + camera.f * p_cam[:, 1] / p_cam[:, 2]
    spec = np.array(camera.ground_to_pixel(forward, left))
    print(f"largest pixel difference over 200 road points: {np.max(np.abs(np.stack([u, v]) - spec)):.2e} px")
except NotImplementedError:
    print("Write T_ego_cam above first, then run this cell again.")

## 5. Points move, directions only turn

A point takes w = 1: the 4×4 turns it and shifts it. A direction takes w = 0: it only turns (lecture: "Points move,
directions only turn").

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/point_vs_direction.png" alt="The mount from the car&#x27;s right side. Drawn from the lens, the direction (0.9962, 0, −0.0872) dips 5 degrees below level and meets the road ahead. The point (0.9962, 0, 1.3128) sits 1 m out along that axis. Read as a direction from the lens instead, it climbs 0.92 rad above level and never comes down.">
<figcaption><em>Lecture figure 11. The mount from the car&#x27;s right side. Drawn from the lens, the direction (0.9962, 0, −0.0872) dips 5 degrees below level and meets the road ahead. The point (0.9962, 0, 1.3128) sits 1 m out along that axis. Read as a direction from the lens instead, it climbs 0.92 rad above level and never comes down.</em></figcaption>
</figure>

**Problem 3.** Write `transform_points(T, p)` (w = 1) and `transform_directions(T, v)` (w = 0) for a 3-vector, using
the 3×3 block `T[:3, :3]` and the last column `T[:3, 3]`.

**Guess first:** with the mount `T`, where does the lens axis (0, 0, 1) go as a direction, and where as a point?

In [ ]:
def transform_points(T, p):
    return np.asarray(p, dtype=float)  # your code: turn AND shift


def transform_directions(T, v):
    return np.asarray(v, dtype=float)  # your code: turn only


T = np.eye(4)
T[:3, :3] = rot_y(pitch) @ B.T
T[:3, 3] = (0.0, 0.0, 1.4)  # this lab's own T, so Problem 3 runs before your T_ego_cam is done
practice("the lens (camera origin) in ego numbers", transform_points(T, (0, 0, 0)), (0.0, 0.0, 1.4), tol=1e-12)
practice("the lens axis as a direction, in ego numbers", transform_directions(T, (0, 0, 1)), (0.9962, 0.0, -0.0872), tol=1e-4)
practice("the lens axis wrongly as a point (w = 1)", transform_points(T, (0, 0, 1)), (0.9962, 0.0, 1.3128), tol=1e-4)


def row_of(p_cam):
    return camera.cy + camera.f * p_cam[1] / p_cam[2]


practice("forward as a direction lands on the horizon row", row_of(invert_pose(T)[:3, :3] @ x), 75.5, tol=0.01)
practice("forward wrongly as a point (w = 1) lands in row", row_of((invert_pose(T) @ np.append(x, 1.0))[:3]), 276.59, tol=0.01)

As a direction the axis dips 5° below level; as a point it sits 1.3128 m up, which is nonsense for something that is
only a direction. Forward as a direction lands on the horizon, row 75.5; treated as a point it lands in row 276.6, off
the bottom of the picture.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/horizon_w.png" alt="A camera picture of the empty road with two marks on the middle column. A dashed line at row 75.5, labelled w = 0, is where the car&#x27;s forward direction lands, the horizon. An arrow off the bottom edge, labelled w = 1, points to row 276.6, where the same direction lands when it is wrongly shifted by the mount.">
<figcaption><em>Lecture figure 12. A camera picture of the empty road with two marks on the middle column. A dashed line at row 75.5, labelled w = 0, is where the car&#x27;s forward direction lands, the horizon. An arrow off the bottom edge, labelled w = 1, points to row 276.6, where the same direction lands when it is wrongly shifted by the mount.</em></figcaption>
</figure>

## 6. Where a pixel's ray meets the road

The ray through pixel (u, v) is ((u − cx) / f, (v − cy) / f, 1) in camera numbers. Turn it into ego numbers
with `w = 0`, then walk λ steps from the lens until z = 0: `λ = −t_z / r_z` (lecture: "Where a pixel's ray meets the
road"). If `r_z ≥ 0` the ray never meets the road: the answer is `nan`.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/pixel_ray.png" alt="The ray from the lens through the paint pixel, (254.5, 150), in two stacked panels. Top, from the car&#x27;s right side: it leaves the lens 1.4 m up and meets the road 2.907 m ahead. Bottom, from above: it heads to the right of straight ahead and lands 1.792 m right, on the white line. Drawn with it, the ray through pixel (159.85, 150) lands at (2.9073, −0.0066).">
<figcaption><em>Lecture figure 13. The ray from the lens through the paint pixel, (254.5, 150), in two stacked panels. Top, from the car&#x27;s right side: it leaves the lens 1.4 m up and meets the road 2.907 m ahead. Bottom, from above: it heads to the right of straight ahead and lands 1.792 m right, on the white line. Drawn with it, the ray through pixel (159.85, 150) lands at (2.9073, −0.0066).</em></figcaption>
</figure>

**Guess first:** the white line's pixel (254.5, 150): about how many steps λ down the ray to the road, and how far
ahead and to the right is the road point?

In [ ]:
ray = pixel_ray(camera, 254.5, 150)
print("ray in camera numbers:", np.round(ray, 4))
r = transform_directions(T, ray)
print("ray in ego numbers:   ", np.round(r, 4))
print("steps to the road:    ", round(float(-T[2, 3] / r[2]), 3))

**Problem 4 and Exercise C.** Write `ray_to_ground(T_ego_cam, ray_cam)` below; it returns the road point as a 3-vector
(forward, left, up) and `nan` when the ray never meets the road. The check sends 200 seeded pixels below the horizon
to the road and compares with `Camera.pixel_to_ground` within $10^{-9}$ m at the simulator's mount and at two others (pitch +1°, height +0.1 m), so use the `T_ego_cam` you are given, then 200 pixels above the horizon, which must
come back `nan`. The worked pixel gives (2.9073, −1.7921).

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/rows_across.png" alt="Every pixel of rows 179, 150, 125, 115 and 95 sent to the road through ray_to_ground, seen from above. Each row lands on a straight line across the road at one distance: 2.058 m, 2.907 m, 4.438 m, 5.592 m and 11.454 m ahead. The lines get wider with distance: row 150 spans 3.018 m to either side, row 95 spans 11.532 m.">
<figcaption><em>Lecture figure 14. Every pixel of rows 179, 150, 125, 115 and 95 sent to the road through ray_to_ground, seen from above. Each row lands on a straight line across the road at one distance: 2.058 m, 2.907 m, 4.438 m, 5.592 m and 11.454 m ahead. The lines get wider with distance: row 150 spans 3.018 m to either side, row 95 spans 11.532 m.</em></figcaption>
</figure>

In [ ]:
from zero2fsd.geom import transform_directions  # swap in your own from Problem 3


def ray_to_ground(T_ego_cam, ray_cam):
    """Where a ray from the camera's lens meets the road.

    Input:   T_ego_cam, the 4 by 4 mount; ray_cam, the ray in camera numbers, shape (3,).
    Output:  the road point in ego numbers, shape (3,): (forward, left, 0); math.nan for all three
             when the ray never meets the road (its ego z is not negative).
    Example: ray_to_ground(T_ego_cam(1.4, math.radians(5)), pixel_ray(Camera(), 254.5, 150))
             -> [2.9073, -1.7921, 0.0]
    """
    raise NotImplementedError

**Guess first:** what does a correct `ray_to_ground` return for a pixel above row 75.5?

In [ ]:
check("1.1.5.c", ray_to_ground)

Every pixel below the horizon lands within $10^{-9}$ m of the simulator's answer (the reference gets $2\times10^{-16}$ m). Treat the ray as a point (`w = 1`) and
all 200 come back `nan`, because the shifted ray no longer points down.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/rows_pitch.png" alt="The distance ahead that each row sees in the middle column, with the camera tipped 5 degrees down, as specified, and 5 degrees up. Row 150 sees 2.907 m with the spec and 4.976 m tipped up. Tipped up, the horizon drops from row 75.5 to row 103.5. The far band, rows 95 to 115, should see 5.59 m to 11.45 m; tipped up, only its rows from 107 to 115 see road, from 70 m, the end of the drawn road, down to 19.75 m.">
<figcaption><em>Lecture figure 15. The distance ahead that each row sees in the middle column, with the camera tipped 5 degrees down, as specified, and 5 degrees up. Row 150 sees 2.907 m with the spec and 4.976 m tipped up. Tipped up, the horizon drops from row 75.5 to row 103.5. The far band, rows 95 to 115, should see 5.59 m to 11.45 m; tipped up, only its rows from 107 to 115 see road, from 70 m, the end of the drawn road, down to 19.75 m.</em></figcaption>
</figure>

## 7. A camera mounted wrong

Now the payoff: put your mount into the simulator's camera and drive. Tip the camera 5° **up** instead of down and
the picture still looks like a road; whether the car notices depends on the road (lecture: "A camera mounted
wrong").

**Guess first:** tip the camera up, and which road shows it: `straight`, `gentle`, both, or neither?

In [ ]:
try:
    for road, mounted, label in ((r, T_ego_cam(1.4, p), name) for r in ("straight", "gentle") for p, name in ((pitch, "tipped down 5 deg (correct)"), (-pitch, "tipped UP 5 deg"))):
        tel = run(Car(), road, camera=Camera(T_ego_cam=mounted))
        print(f"{road}, {label}: completed {tel.completed}, ended after {tel.progress_m[-1]:.1f} m, mean distance from the lane centre {np.mean(np.abs(tel.lat)):.3f} m")
except NotImplementedError:
    print("Write T_ego_cam above first, then run this cell again.")

The correct mount completes `gentle` with a mean distance from the lane centre of 0.376 m, step for step the same 1778 steps as with the simulator's own camera. Tipped up, the car leaves the lane at 86.3 m, before
the first bend: the far band places road 20 to 70 m ahead only 5.6 to 7.1 m ahead, so the bend reads as a sharp turn.
On `straight` the same mistake drives to the end (mean 0.004 m).

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/lat_pitch.png" alt="Distance from the lane centre against distance along the road for the provided car with the renderer&#x27;s camera tipped 5 degrees down, level and 5 degrees up, on straight (top) and on gentle (bottom). Same car, same seed, same speed. On straight all three finish. On gentle the tipped-up camera leaves the lane to the left at 86.3 m, before the first bend starts at 100 m, and the level camera leaves at 123.0 m.">
<figcaption><em>Lecture figure 16. Distance from the lane centre against distance along the road for the provided car with the renderer&#x27;s camera tipped 5 degrees down, level and 5 degrees up, on straight (top) and on gentle (bottom). Same car, same seed, same speed. On straight all three finish. On gentle the tipped-up camera leaves the lane to the left at 86.3 m, before the first bend starts at 100 m, and the level camera leaves at 123.0 m.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/heading_pitch.png" alt="The box&#x27;s heading for a car held on the lane centre of gentle and pointing along it, from 20 m to 90 m along. With the spec camera it stays within 0.002 rad of 0. With the camera tipped 5 degrees up it falls to −0.035 rad at 60 m, −0.078 rad at 70 m and −0.286 rad at 90 m: the box says the car points right of the lane when it points straight along it. The first bend starts at 100 m.">
<figcaption><em>Lecture figure 17. The box&#x27;s heading for a car held on the lane centre of gentle and pointing along it, from 20 m to 90 m along. With the spec camera it stays within 0.002 rad of 0. With the camera tipped 5 degrees up it falls to −0.035 rad at 60 m, −0.078 rad at 70 m and −0.286 rad at 90 m: the box says the car points right of the lane when it points straight along it. The first bend starts at 100 m.</em></figcaption>
</figure>

Now the swap for grading: your `world_to_ego` and `T_ego_cam` go into the camera and the provided car drives `gentle`. Your pair is checked first, then the
car drives. Expect 0.376 m: that is the provided car's own error on `gentle` with the simulator's camera, step for step the
same 1778 steps; a correct mount adds nothing to it. The pass line also says no camera pictures differ.

In [ ]:
from zero2fsd.geom import world_to_ego  # swap in your own from *The view from the driver's seat*

camera_mount = (world_to_ego, T_ego_cam)  # the pair the simulator draws a camera picture through

In [ ]:
check("1.1.5.b", camera_mount)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/gate_breaks.png" alt="Distance from the lane centre against distance along gentle for three mounts written as code and swapped into the renderer: your T_ego_cam, which finishes with a mean of 0.376 m; the same with the sign of φ flipped, which leaves at 86.3 m; and rot_x in place of rot_y, which leaves at 119.1 m. The 0.376 m is the provided car&#x27;s own error: the simulator&#x27;s own camera gives the same 1778 steps and the same 0.376 m.">
<figcaption><em>Lecture figure 18. Distance from the lane centre against distance along gentle for three mounts written as code and swapped into the renderer: your T_ego_cam, which finishes with a mean of 0.376 m; the same with the sign of φ flipped, which leaves at 86.3 m; and rot_x in place of rot_y, which leaves at 119.1 m. The 0.376 m is the provided car&#x27;s own error: the simulator&#x27;s own camera gives the same 1778 steps and the same 0.376 m.</em></figcaption>
</figure>

## 8. Break it, fix it

A teammate says "a camera 0.2 m too low is the same error, just smaller". Mount the camera at 1.2 m and at 1.6 m,
but keep the box's own 1.4 m, and drive `gentle`.

**Guess first:** does the car leave the road, wobble, or complete with a bigger error? Which of heading and distance
does a wrong height damage?

In [ ]:
try:
    for height in (1.2, 1.6):
        tel = run(Car(), "gentle", camera=Camera(T_ego_cam=T_ego_cam(height, pitch)))
        print(f"camera at {height} m: completed {tel.completed}, mean distance from the lane centre {np.mean(np.abs(tel.lat)):.3f} m")
except NotImplementedError:
    print("Write T_ego_cam above first, then run this cell again.")

<details><summary>Hint (open after you have a guess)</summary>

Both complete: 1.2 m gives 0.286 m and 1.6 m gives 0.472 m. A height error only scales every distance by 1.4 / h, so
the box misjudges how far ahead things are and how wide the lane is, but angles survive. A pitch error shifts every
ray's angle, and that is why it is the one that breaks the box.

</details>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/height_scale.png" alt="The ray through row 150&#x27;s middle column, from the car&#x27;s right side, drawn from two lens heights at the same angle, 0.4488 rad below level. From a lens truly 1.2 m up it meets the road 2.492 m ahead. From the 1.4 m the box assumes it meets the road 2.907 m ahead. The two triangles have the same shape, so 2.907 is 7/6 of 2.492, as 1.4 is 7/6 of 1.2.">
<figcaption><em>Lecture figure 19. The ray through row 150&#x27;s middle column, from the car&#x27;s right side, drawn from two lens heights at the same angle, 0.4488 rad below level. From a lens truly 1.2 m up it meets the road 2.492 m ahead. From the 1.4 m the box assumes it meets the road 2.907 m ahead. The two triangles have the same shape, so 2.907 is 7/6 of 2.492, as 1.4 is 7/6 of 1.2.</em></figcaption>
</figure>

## What's next

Every lap so far ran at 8 m/s. Who chose it, and how would the car change speed? That is the next module,
*Longitudinal control and vehicle motion*.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/next_lesson.png" alt="The provided car&#x27;s speed through one lap of gentle, plotted against time: a flat line at 8 m/s from the first step to the last, 88.9 s later.">
<figcaption><em>Lecture figure 20. The provided car&#x27;s speed through one lap of gentle, plotted against time: a flat line at 8 m/s from the first step to the last, 88.9 s later.</em></figcaption>
</figure>

## Stretch: the challenge problems

Optional, not graded; the answers are in the lecture (lecture: "Challenge problems"). Four marks painted on the road, at (3, ±1)
and (8, ±1) m, let you recover the camera's pitch and height from one picture, with neither known.

**Guess first:** the near marks sit in row 147.789 and the far marks in row 103.291. If a mark found one pixel too low in every row
moves your pitch estimate, by about how much: 0.0001 rad, 0.007 rad, or 0.05 rad?

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/calibration_marks.png" alt="The empty road as the spec camera sees it, with the four painted marks circled. The near pair, 3 m ahead and 1 m to either side, sits in row 147.789, at columns 108.063 and 210.937. The far pair, 8 m ahead, sits in row 103.291, at columns 139.726 and 179.274.">
<figcaption><em>Lecture figure 21. The empty road as the spec camera sees it, with the four painted marks circled. The near pair, 3 m ahead and 1 m to either side, sits in row 147.789, at columns 108.063 and 210.937. The far pair, 8 m ahead, sits in row 103.291, at columns 139.726 and 179.274.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.5-the-cameras-point-of-view/figures/calibration_spread.png" alt="The pitch recovered by 500 calibrations from the four marks, each with random row errors of standard deviation 0.5 px, as a histogram. They centre on 0.0877 rad, next to the true 0.0873 rad, with a standard deviation of 0.0044 rad, 0.7 of one row&#x27;s angle.">
<figcaption><em>Lecture figure 22. The pitch recovered by 500 calibrations from the four marks, each with random row errors of standard deviation 0.5 px, as a histogram. They centre on 0.0877 rad, next to the true 0.0873 rad, with a standard deviation of 0.0044 rad, 0.7 of one row&#x27;s angle.</em></figcaption>
</figure>

Then try the others: why no pixel ray points down when you give it `w = 1`, and why a camera 1.2 m up read as 1.4 m
scales every road point by the same factor.